# Misc.json Generation — Norte Amazónica clusters

This notebook generates one `Misc.json` file per cluster (C1 to C5) for EnergyScope Norte Amazónica.

Each `Misc.json` contains the constraints and parameters passed to EnergyScope for a given cluster:
- `solar_area` : available land for solar PV (km²), computed as **rooftop + ground** per cluster
- mobility / freight / heating shares : fixed to observed values for this region
- flights : only for clusters with active airports (C3 and C5)

> **Thesis improvement:** Pablo's model used a single value (47 764 km²) for the whole SA-NO region,
> regardless of cluster size and without land-use constraints.
> This notebook replaces it with a two-component methodology (rooftop PV + ground PV) detailed in section 2.

In [7]:
import json
import os
import pandas as pd

OUTPUT_DIR = "output_energyscope"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CLUSTERS = {
    1: ["Exaltación", "Reyes", "Santa_Rosa_Beni", "Ixiamas"],
    2: ["Bolpebra"],
    3: ["Guayaramerín", "Riberalta", "Puerto_Gonzalo_Moreno"],
    4: ["Bella_Flor", "Filadelfia", "Ingavi", "Nueva_Esperanza", "Porvenir",
        "Puerto_Rico", "San_Lorenzo", "San_Pedro", "Santa_Rosa_Pando",
        "Santos_Mercado", "Sena", "Villa_Nueva"],
    5: ["Cobija"],
}

## 1. Read area and households from CSV_final

Two columns are read from `CSV_final.csv` for each municipality:
- `SUPERFICIE (km²)` → land area
- `NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD | 2024 | Total` → total number of households (2024 census)

Municipality names in the CSV use spaces (e.g. `Santa Rosa`) while cluster names use underscores (`Santa_Rosa_Beni`). The two municipalities both named `Santa Rosa` are disambiguated using the `DEPARTAMENTO` column.

In [8]:
df = pd.read_csv("../exctraction of data/output/CSV_final.csv")

# Keep only municipality rows (drop department/province aggregate rows)
df = df[df["MUNICIPIO/TIOC"].notna() & (df["MUNICIPIO/TIOC"] != "")].copy()

HH_COL = "NÚMERO DE VIVIENDAS POR FUENTE DE ELECTRICIDAD | 2024 | Total"

AREA       = {}
HOUSEHOLDS = {}
for _, row in df.iterrows():
    key  = row["MUNICIPIO/TIOC"].replace(" ", "_")
    dept = row["DEPARTAMENTO"]
    # "Santa Rosa" appears in both Beni and Pando → add department to disambiguate
    if key == "Santa_Rosa":
        key = f"Santa_Rosa_{dept}"   # → "Santa_Rosa_Beni" or "Santa_Rosa_Pando"
    AREA[key]       = row["SUPERFICIE (km²)"]
    HOUSEHOLDS[key] = row[HH_COL]

## 2. Solar area calculation — thesis improvement

**Why we depart from Pablo's model:**  
Pablo's model assigned a single value of 47 764.39 km² to the entire Norte Amazónica region (SA-NO), regardless of cluster size. This value was derived from atlite without any land-use constraint, and was explicitly flagged as a limitation in the paper's conclusion section. Splitting it uniformly across clusters would perpetuate this flaw.

**New methodology — two components:**

**Rooftop PV** (proportional to buildings):
$$\text{solar\_area\_rooftop}_k = N_{\text{hh},k} \times 40 \times 10^{-6} \text{ km}^2/\text{hh}$$
- Average Bolivian rural building footprint: ~80 m²
- Usable rooftop fraction: 50% (shading, slope, structural constraints)
- → 80 m² × 0.50 = **40 m²/household** usable for PV

**Ground-mounted PV** (proportional to land area):
$$\text{solar\_area\_ground}_k = \text{area}_k \times 0.01$$
- Norte Amazónica is ~90% Amazon rainforest — utility-scale solar is only realistic on degraded land, existing clearings, and riverbanks
- **1%** is a conservative estimate consistent with land-use constraints missing from Pablo's model

**High-irradiance ground:** `solar_area_ground_high_irr = 0`  
Norte Amazónica irradiation is well below 1 800 kWh/m²/yr (the threshold for CSP / high-irradiance classification).

> This methodology is documented as a thesis improvement over the reference model.

In [9]:
cluster_area = {k: sum(AREA[m]       for m in munis) for k, munis in CLUSTERS.items()}
cluster_hh   = {k: sum(HOUSEHOLDS[m] for m in munis) for k, munis in CLUSTERS.items()}

# Rooftop PV: 80 m² building footprint × 50% usable = 40 m²/hh = 40e-6 km²/hh
solar_area_rooftop = {k: round(cluster_hh[k]   * 40e-6, 4) for k in CLUSTERS}

# Ground PV: 1% of cluster land area (only degraded land / clearings available)
solar_area_ground  = {k: round(cluster_area[k] * 0.01,  4) for k in CLUSTERS}

## 3. Misc parameters

**`BASE`** — values common to all clusters (from Pablo's Pando SA-NO reference `Misc.json`):
- `import_capacity = 2` : Big-M upper bound for EnergyScope (not a physical limit)
- `share_freight_train = 0` : no rail infrastructure in Norte Amazónica
- `share_heat_dhn = 0` : no district-heating network in this region
- `share_dispersion = 0` : optimizer freely allocates between Home Systems and community scale; can be raised toward 0.762 (Roger's lowlands value) in sensitivity analysis

**`FLIGHTS`** — short-haul aviation share per cluster:
- C3 (Riberalta + Guayaramerín) : two regional airports in operation
- C5 (Cobija) : Aasana international airport
- C1, C2, C4 : no airports → 0

In [ ]:
BASE = {
    "re_share_primary":           0,
    "gwp_limit":                  1e15,
    "import_capacity":            2,
    "solar_area_ground_high_irr": 0,
    "share_mobility_public_min":  0.729,
    "share_mobility_public_max":  0.72989,
    "share_freight_train_min":    0,
    "share_freight_train_max":    0,
    "share_freight_road_min":     0.978,
    "share_freight_road_max":     0.97841,
    "share_freight_boat_min":     0.021,
    "share_freight_boat_max":     0.02160,
    "share_heat_dhn_min":         0.00,
    "share_heat_dhn_max":         0.00,
    "share_dispersion":           0,
    "share_ned":                  {"HVC": 1.00, "METHANOL": 0.00, "AMMONIA": 0.00},
    # share_dispersion = 0: optimizer freely decides how much electricity
    # comes from Home Systems (PV_HS, HS_DIESEL, BATT_HS) vs community scale
    # Consistent with Pablo's SA-xx approach (SA-PA, SA-BN all = 0)
    # Can be increased toward 0.762 (Roger's lowlands value) in sensitivity analysis
}

# (min, max) short-haul flight shares per cluster
FLIGHTS = {
    1: (0.0,   0.0),
    2: (0.0,   0.0),
    3: (0.030, 0.03006),   # Riberalta + Guayaramerín airports
    4: (0.0,   0.0),
    5: (0.030, 0.03006),   # Cobija international airport
}

## 4. Generate one Misc.json per cluster

For each cluster, the dict is built by combining `BASE`, the cluster-specific `solar_area`, and the flight shares. The file is saved to `output_energyscope/C{k}/Misc.json`.

In [ ]:
for k in sorted(CLUSTERS):
    flights_min, flights_max = FLIGHTS[k]

    misc = {
        "re_share_primary":             BASE["re_share_primary"],
        "gwp_limit":                    BASE["gwp_limit"],
        "import_capacity":              BASE["import_capacity"],
        "solar_area_rooftop":           solar_area_rooftop[k],
        "solar_area_ground":            solar_area_ground[k],
        "solar_area_ground_high_irr":   BASE["solar_area_ground_high_irr"],
        "share_mobility_public_min":    BASE["share_mobility_public_min"],
        "share_mobility_public_max":    BASE["share_mobility_public_max"],
        "share_short_haul_flights_min": flights_min,
        "share_short_haul_flights_max": flights_max,
        "share_freight_train_min":      BASE["share_freight_train_min"],
        "share_freight_train_max":      BASE["share_freight_train_max"],
        "share_freight_road_min":       BASE["share_freight_road_min"],
        "share_freight_road_max":       BASE["share_freight_road_max"],
        "share_freight_boat_min":       BASE["share_freight_boat_min"],
        "share_freight_boat_max":       BASE["share_freight_boat_max"],
        "share_heat_dhn_min":           BASE["share_heat_dhn_min"],
        "share_heat_dhn_max":           BASE["share_heat_dhn_max"],
        "share_ned":                    dict(BASE["share_ned"]),
        "share_dispersion":             BASE["share_dispersion"],
    }

    out_path = f"{OUTPUT_DIR}/C{k}/Misc.json"
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(misc, f, indent=4)
        f.write("\n")

print(f"Saved Misc.json files")

Saved Misc.json files


## 5. Verification

In [14]:
print(f"{'Cluster':<8} {'Area (km2)':>12} {'Households':>12} {'Rooftop (km2)':>15} {'Ground (km2)':>14} {'Total (km2)':>13}")
print("-" * 78)
for k in sorted(CLUSTERS):
    print(f"  C{k:<6} {cluster_area[k]:>12,} {cluster_hh[k]:>12,.0f} "
          f"{solar_area_rooftop[k]:>15.4f} {solar_area_ground[k]:>14.4f}")
print("-" * 78)
print(f"  {'TOTAL':<6} {sum(cluster_area.values()):>12,} {sum(cluster_hh.values()):>12,.0f} "
      f"{sum(solar_area_rooftop.values()):>15.4f} {sum(solar_area_ground.values()):>14.4f}")

Cluster    Area (km2)   Households   Rooftop (km2)   Ground (km2)   Total (km2)
------------------------------------------------------------------------------
  C1            65,316       10,933          0.4373       653.1600
  C2             7,000          802          0.0321        70.0000
  C3            21,482       40,298          1.6119       214.8200
  C4            51,772       16,612          0.6645       517.7200
  C5               740       15,564          0.6226         7.4000
------------------------------------------------------------------------------
  TOTAL       146,310       84,209          3.3684      1463.1000
